<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/corrected_feat_creator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# Causally-Corrected Feature Engineering Pipeline
# Notebook Version (Phase 2 Thesis)
# ============================================================

In [ ]:
!pip install polars

In [ ]:
# ============================================================
# 0. Imports
# ============================================================

import polars as pl
import duckdb
import numpy as np
from pathlib import Path

pl.Config.set_tbl_rows(10)

polars.config.Config

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

BASE = "/content/data/"
OUTPUT_DIR = "/content/output/"

Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

WORKLOAD_CAP = 20
GRID_SIZE = 500
ROLLING_DAYS = 7

In [3]:
# ============================================================
# 2. Load Dataset
# ============================================================

delivery = (
    pl.read_csv(BASE + "delivery_data.csv")
    .with_columns([
        pl.col("receipt_time").str.to_datetime(),
        pl.col("sign_time").str.to_datetime(),
    ])
    .rename({"horizon_ETA": "eta_mins"})
)

print(delivery.shape)
delivery.head()

NameError: name 'BASE' is not defined

In [4]:
# ============================================================
# 3. Basic ETA Filtering
# ============================================================

delivery = delivery.filter(
    (pl.col("eta_mins") > 0) &
    (pl.col("eta_mins") < 1440)
)

print(delivery.shape)

NameError: name 'delivery' is not defined

Batching

In [5]:
# ============================================================
# 5. Batch Features
# ============================================================

delivery = delivery.sort(
    ["delivery_user_id", "receipt_time", "order_id"]
)

# Batch size
delivery = delivery.with_columns(
    pl.len().over(
        ["delivery_user_id", "receipt_time"]
    ).alias("batch_size")
)

# Dispatch rank
delivery = delivery.with_columns(
    pl.int_range(0, pl.len())
    .over(["delivery_user_id", "receipt_time"])
    .alias("batch_rank_dispatch")
)

# Actual delivery rank (post-hoc)
delivery = delivery.with_columns(
    (
        pl.col("sign_time")
        .rank("ordinal")
        .over(["delivery_user_id", "receipt_time"]) - 1
    ).alias("batch_rank_actual")
)

delivery.select([
    "batch_size",
    "batch_rank_dispatch",
    "batch_rank_actual"
]).head()


NameError: name 'delivery' is not defined

In [ ]:
# ============================================================
# 6. Explicit Batch ID
# ============================================================

delivery = delivery.with_columns(
    (
        pl.col("delivery_user_id") + "__" +
        pl.col("receipt_time")
        .dt.epoch("s")
        .cast(pl.Utf8)
    ).alias("batch_id")
)

delivery.select("batch_id").head()

In [ ]:
# ============================================================
# 7. STRICT CAUSAL WORKLOAD
# ============================================================

delivery = delivery.sort(
    ["delivery_user_id", "receipt_time"]
)

start_events = delivery.select([
    "delivery_user_id",
    pl.col("receipt_time").alias("time"),
    pl.lit(1).alias("delta"),
    pl.lit(1).alias("priority"),
])

end_events = delivery.select([
    "delivery_user_id",
    pl.col("sign_time").alias("time"),
    pl.lit(-1).alias("delta"),
    pl.lit(2).alias("priority"),
])

events = (
    pl.concat([start_events, end_events])
    .sort(["delivery_user_id", "time", "priority"])
    .with_columns(
        pl.col("delta")
        .cum_sum()
        .over("delivery_user_id")
        .alias("active_orders")
    )
)

delivery = delivery.join_asof(
    events.select([
        "delivery_user_id",
        "time",
        "active_orders"
    ]),
    left_on="receipt_time",
    right_on="time",
    by="delivery_user_id",
    strategy="backward",
)

delivery = delivery.with_columns(
    (
        pl.col("active_orders") - 1
    )
    .clip(lower_bound=0)
    .alias("workload_causal")
)

delivery.select("workload_causal").describe()

Temporal features

In [ ]:
# ============================================================
# 9. Temporal Features
# ============================================================

delivery = delivery.with_columns([

    pl.col("receipt_time").dt.hour().alias("hour"),
    pl.col("receipt_time").dt.weekday().alias("weekday"),
    pl.col("receipt_time").dt.date().alias("receipt_date"),

])

delivery = delivery.with_columns([

    (2 * np.pi * pl.col("hour") / 24)
    .sin()
    .alias("hour_sin"),

    (2 * np.pi * pl.col("hour") / 24)
    .cos()
    .alias("hour_cos"),

    (2 * np.pi * pl.col("weekday") / 7)
    .sin()
    .alias("day_sin"),

    (2 * np.pi * pl.col("weekday") / 7)
    .cos()
    .alias("day_cos"),

])

In [ ]:
# ============================================================
# 10. Delivery Sequence (Per-Day)
# ============================================================

delivery = delivery.with_columns(

    pl.col("order_id")
    .cum_count()
    .over([
        "delivery_user_id",
        "receipt_date"
    ])
    .alias("delivery_sequence_daily")

)


In [ ]:
# ============================================================
# 11. Spatial Congestion
# ============================================================

delivery = delivery.with_columns([

    (pl.col("receipt_lng") // GRID_SIZE)
    .alias("grid_x"),

    (pl.col("receipt_lat") // GRID_SIZE)
    .alias("grid_y"),

    pl.col("receipt_time")
    .dt.truncate("1h")
    .alias("time_window"),

])

# Daily congestion
sci_daily = (

    delivery
    .group_by([
        "grid_x",
        "grid_y",
        "receipt_date",
        "time_window"
    ])
    .len()
    .rename({
        "len": "spatial_congestion_daily"
    })

)

delivery = delivery.join(
    sci_daily,
    on=[
        "grid_x",
        "grid_y",
        "receipt_date",
        "time_window"
    ],
    how="left"
)


In [ ]:
# ============================================================
# 13. GPS Missingness
# ============================================================

delivery = delivery.with_columns(

    pl.col("speed_mean")
    .is_not_null()
    .cast(pl.Int8)
    .alias("is_trajectory_available")

)

In [ ]:
# ============================================================
# 15. Batch-Level Aggregation
# ============================================================

batch_df = (

    delivery
    .group_by("batch_id")
    .agg([

        pl.mean("eta_mins")
        .alias("eta_mins_mean"),

        pl.mean("workload_causal")
        .alias("workload_mean"),

        pl.mean("pickup_destination_distance")
        .alias("distance_mean"),

        pl.first("batch_size")
        .alias("batch_size"),

    ])

)

batch_df.head()


In [ ]:
# ============================================================
# 16. Sanity Checks
# ============================================================

assert delivery["eta_mins"].null_count() == 0

assert delivery.height == \
       delivery["order_id"].n_unique()

assert delivery["workload_causal"].min() >= 0

print("Sanity checks passed")

# ============================================================
# 17. Save Outputs
# ============================================================

delivery.write_parquet(
    OUTPUT_DIR + "delivery_features.parquet"
)

batch_df.write_parquet(
    OUTPUT_DIR + "batch_features.parquet"
)

print("Saved outputs")

# ============================================================
# 18. Next Stage
# ============================================================

"""
Next pipeline stages:

1. PCMCI+
2. CD-NOD
3. ICP / IRM
4. Invariant feature extraction
5. Contextual bandit state construction
6. Offline policy evaluation
